# Inference Method as a Measurement Facet: Benchmark Validity Under Speculative Decoding

**End-to-end reproduction notebook**

This notebook reproduces all results reported in the paper.  
Execution is divided into two stages:

1. **Logit collection** (§3) — requires a GPU and HuggingFace model access.  
   Set `RUN_LOGIT_COLLECTION = True` in §1 to run this stage.  
   Pre-collected `.npz` caches are provided in `cache/logits/` for full reproduction.

2. **Analysis** (§4–§13) — runs entirely from the saved logit caches; no GPU required.

**Benchmarks:** 500 MMLU items · 500 HellaSwag items · 200 GSM8K items  
**Model pairs:** Qwen2.5-0.5B → Qwen2.5-7B · Llama-3.2-1B → Llama-3.1-8B  
**Outputs:** All CSVs and figures are written to `results/`

In [ ]:
# § 0  Dependencies
# Run once to install required packages.
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers", "accelerate", "datasets",
    "torch", "numpy", "scipy", "pandas",
    "matplotlib", "seaborn", "tqdm", "statsmodels", "scikit-learn"
])

## § 1  Configuration
All tunable constants live here. To re-collect logits from scratch, set `RUN_LOGIT_COLLECTION = True`.

In [ ]:
import os, re, gc, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import pearsonr, pointbiserialr
from scipy.optimize import curve_fit
import statsmodels.api as sm
from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
CACHE_DIR  = Path("cache")
LOGIT_DIR  = CACHE_DIR / "logits"
RESULT_DIR = Path("results")
for d in [LOGIT_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Model pairs: (draft_short, draft_hf_id, target_short, target_hf_id) ──────
MODEL_PAIRS = [
    ("Qwen2.5-0.5B", "Qwen/Qwen2.5-0.5B-Instruct",
     "Qwen2.5-7B",   "Qwen/Qwen2.5-7B-Instruct"),
    ("Llama-3.2-1B", "meta-llama/Llama-3.2-1B-Instruct",
     "Llama-3.1-8B", "meta-llama/Llama-3.1-8B-Instruct"),
]

# ── Item counts ───────────────────────────────────────────────────────────────
N_GSM8K = 200   # GSM8K items loaded directly from the test split
N_FULL  = 500   # items for MMLU and HellaSwag

# ── Logit-collection settings ─────────────────────────────────────────────────
RUN_LOGIT_COLLECTION = False   # Set True to (re-)collect logits (needs GPU)
TOP_K_STORE          = 100     # top-k logprobs stored per generation step
COLLECTION_TEMP      = 0.7     # temperature used during logit collection
LOGIT_CACHE_VERSION  = 2       # bump to invalidate stale caches
N_RUNS               = 1       # generation runs per model-benchmark pair
MAX_NEW_TOKENS = {"gsm8k": 384, "mmlu": 1, "hellaswag": 1}

# ── Decoding conditions ───────────────────────────────────────────────────────
# epsilon=None  → baseline (condition A) or temperature replay (T_*)
# epsilon=0.0   → exact speculative decoding (condition B)
# epsilon>0     → approximate speculation: accept if ratio >= (1 - epsilon)
CONDITIONS = {
    "A":     {"type": "spec", "epsilon": None, "temperature": COLLECTION_TEMP},
    "B":     {"type": "spec", "epsilon": 0.00, "temperature": COLLECTION_TEMP},
    "C_090": {"type": "spec", "epsilon": 0.10, "temperature": COLLECTION_TEMP},
    "C_080": {"type": "spec", "epsilon": 0.20, "temperature": COLLECTION_TEMP},
    "C_070": {"type": "spec", "epsilon": 0.30, "temperature": COLLECTION_TEMP},
    "C_060": {"type": "spec", "epsilon": 0.40, "temperature": COLLECTION_TEMP},
    "T_030": {"type": "temp", "epsilon": None, "temperature": 0.3},
    "T_050": {"type": "temp", "epsilon": None, "temperature": 0.5},
    "T_070": {"type": "temp", "epsilon": None, "temperature": 0.7},
    "T_100": {"type": "temp", "epsilon": None, "temperature": 1.0},
    "T_150": {"type": "temp", "epsilon": None, "temperature": 1.5},
}
SPEC_CONDITIONS = [k for k, v in CONDITIONS.items() if v["type"] == "spec" and k != "A"]
TEMP_CONDITIONS = [k for k, v in CONDITIONS.items() if v["type"] == "temp"]
ALL_CONDITIONS  = list(CONDITIONS.keys())
N_CONDITIONS_NON_BASELINE = len(ALL_CONDITIONS) - 1  # for Bonferroni multiplier

print("Configuration loaded.")
print(f"  Benchmarks: GSM8K={N_GSM8K}, MMLU={N_FULL}, HellaSwag={N_FULL}")
print(f"  Model pairs: {len(MODEL_PAIRS)}")
print(f"  Conditions:  {len(CONDITIONS)} (including baseline)")
print(f"  Logit collection: {'ENABLED' if RUN_LOGIT_COLLECTION else 'disabled (using cache)'}")

## § 2  Data Loading
Loads benchmark items from HuggingFace. Each item is a dict with keys  
`id`, `prompt`, `answer`, `choices` (MCQ), `format`, `construct`, `benchmark`.

In [ ]:
def load_gsm8k(n: int = N_GSM8K) -> list[dict]:
    """Load n items from the GSM8K test split."""
    ds = load_dataset("openai/gsm8k", "main", split="test")
    ds = ds.shuffle(seed=SEED).select(range(n))
    items = []
    for i, ex in enumerate(ds):
        answer_raw = ex["answer"]
        # Extract the numeric answer after "####"
        match = re.search(r"####\s*([\d,\.\-]+)", answer_raw)
        num_answer = match.group(1).replace(",", "").strip() if match else answer_raw.strip()
        items.append({
            "id":        f"gsm8k_{i}",
            "prompt":    f"Solve the following math problem step by step.\n\nProblem: {ex['question']}\n\nAnswer:",
            "answer":    num_answer,
            "choices":   None,
            "format":    "freeform",
            "construct": "math_reasoning",
            "benchmark": "gsm8k",
        })
    return items


def load_mmlu(n: int = N_FULL) -> list[dict]:
    """Load n items from the MMLU test split (all subjects)."""
    ds = load_dataset("cais/mmlu", "all", split="test")
    ds = ds.shuffle(seed=SEED).select(range(n))
    LETTERS = ["A", "B", "C", "D"]
    items = []
    for i, ex in enumerate(ds):
        choices = ex["choices"]
        prompt_lines = [f"Question: {ex['question']}"]
        for j, c in enumerate(choices):
            prompt_lines.append(f"{LETTERS[j]}. {c}")
        prompt_lines.append("Answer:")
        items.append({
            "id":        f"mmlu_{i}",
            "prompt":    "\n".join(prompt_lines),
            "answer":    LETTERS[int(ex["answer"])],
            "choices":   choices,
            "format":    "mcq",
            "construct": "knowledge_reasoning",
            "benchmark": "mmlu",
        })
    return items


def load_hellaswag(n: int = N_FULL) -> list[dict]:
    """Load n items from the HellaSwag validation split."""
    ds = load_dataset("Rowan/hellaswag", split="validation")
    ds = ds.shuffle(seed=SEED).select(range(n))
    LETTERS = ["A", "B", "C", "D"]
    items = []
    for i, ex in enumerate(ds):
        endings = ex["endings"]
        prompt_lines = [f"Context: {ex['ctx']}\n\nWhich ending best completes the sentence?"]
        for j, e in enumerate(endings):
            prompt_lines.append(f"{LETTERS[j]}. {e}")
        prompt_lines.append("Answer:")
        items.append({
            "id":        f"hellaswag_{i}",
            "prompt":    "\n".join(prompt_lines),
            "answer":    LETTERS[int(ex["label"])],
            "choices":   endings,
            "format":    "mcq",
            "construct": "commonsense_reasoning",
            "benchmark": "hellaswag",
        })
    return items


# Load all items
print("Loading benchmark items ...")
ITEMS = {
    "gsm8k":     load_gsm8k(),
    "mmlu":      load_mmlu(),
    "hellaswag": load_hellaswag(),
}
for bname, items in ITEMS.items():
    print(f"  {bname}: {len(items)} items loaded")

## § 3  Logit Collection
**Requires a GPU and HuggingFace model access.**
Set `RUN_LOGIT_COLLECTION = True` in § 1 to collect fresh logits.
Each cache file is saved to `cache/logits/{model}__{benchmark}__run{n}.npz`.

For each item, we store:
- `gen_ids`: token ids of the generated sequence  
- `topk_ids`, `topk_logprobs`: top-`TOP_K_STORE` token ids and log-probabilities per generation step

In [ ]:
# ── Tokenizer cache ────────────────────────────────────────────────────────────
_tokenizer_cache: dict = {}

def get_tokenizer(model_name: str):
    """Return a cached tokenizer for *model_name*."""
    if model_name not in _tokenizer_cache:
        _tokenizer_cache[model_name] = AutoTokenizer.from_pretrained(
            model_name, use_fast=True
        )
    return _tokenizer_cache[model_name]


def get_logit_cache_path(model_name: str, benchmark: str, run: int = 0) -> Path:
    """Return the path for a logit cache file."""
    safe = model_name.replace("/", "__")
    return LOGIT_DIR / f"{safe}__{benchmark}__run{run}.npz"


def collect_logits(
    model_name: str,
    items: list[dict],
    benchmark: str,
    run: int = 0,
    temperature: float = COLLECTION_TEMP,
    max_new_tokens: int = None,
) -> None:
    """
    Generate token sequences and store top-k logprobs for each item.

    The cache is saved to LOGIT_DIR as a compressed .npz file containing:
      cache_version, item_ids, gen_ids, topk_ids, topk_logprobs

    Args:
        model_name:     HuggingFace model id.
        items:          List of benchmark item dicts.
        benchmark:      Benchmark name (used for MAX_NEW_TOKENS lookup).
        run:            Run index (for multiple independent runs).
        temperature:    Sampling temperature.
        max_new_tokens: Override max new tokens; uses MAX_NEW_TOKENS dict if None.
    """
    cache_path = get_logit_cache_path(model_name, benchmark, run)
    if cache_path.exists():
        print(f"  Cache exists, skipping: {cache_path.name}")
        return

    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("WARNING: no GPU found. Collection will be very slow.")

    if max_new_tokens is None:
        max_new_tokens = MAX_NEW_TOKENS.get(benchmark, 256)

    print(f"  Loading {model_name} onto {device} ...")
    tokenizer = get_tokenizer(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    model.eval()

    rng = torch.Generator(device=device)
    rng.manual_seed(SEED + run)

    all_gen_ids, all_topk_ids, all_topk_logprobs, item_ids_out = [], [], [], []

    for item in tqdm(items, desc=f"{model_name.split('/')[-1]} / {benchmark}"):
        inputs = tokenizer(item["prompt"], return_tensors="pt").to(device)
        prompt_len = inputs["input_ids"].shape[1]

        gen_ids_item, topk_ids_item, topk_lp_item = [], [], []

        with torch.no_grad():
            input_ids = inputs["input_ids"]
            for _ in range(max_new_tokens):
                out = model(input_ids=input_ids)
                logits = out.logits[0, -1, :]           # vocab-size
                lp = torch.log_softmax(logits / temperature, dim=-1)
                topk = torch.topk(lp, TOP_K_STORE)

                # Sample next token
                probs = torch.softmax(logits / temperature, dim=-1)
                next_tok = torch.multinomial(probs, 1, generator=rng).item()

                gen_ids_item.append(next_tok)
                topk_ids_item.append(topk.indices.cpu().numpy().astype(np.int32))
                topk_lp_item.append(topk.values.cpu().numpy().astype(np.float32))

                input_ids = torch.cat(
                    [input_ids, torch.tensor([[next_tok]], device=device)], dim=1
                )

                # Stop at EOS or after first token for MCQ
                if next_tok == tokenizer.eos_token_id or max_new_tokens == 1:
                    break

        all_gen_ids.append(np.array(gen_ids_item, dtype=np.int32))
        all_topk_ids.append(np.stack(topk_ids_item))    # (steps, TOP_K_STORE)
        all_topk_logprobs.append(np.stack(topk_lp_item))
        item_ids_out.append(item["id"])

    # Use object arrays to store variable-length sequences
    gen_arr  = np.empty(len(all_gen_ids), dtype=object)
    tkid_arr = np.empty(len(all_topk_ids), dtype=object)
    tklp_arr = np.empty(len(all_topk_logprobs), dtype=object)
    for i in range(len(all_gen_ids)):
        gen_arr[i]  = all_gen_ids[i]
        tkid_arr[i] = all_topk_ids[i]
        tklp_arr[i] = all_topk_logprobs[i]

    np.savez_compressed(
        cache_path,
        cache_version=np.array(LOGIT_CACHE_VERSION),
        item_ids=np.array(item_ids_out),
        gen_ids=gen_arr,
        topk_ids=tkid_arr,
        topk_logprobs=tklp_arr,
    )
    print(f"  Saved: {cache_path}")

    # Free GPU memory
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


if RUN_LOGIT_COLLECTION:
    print("=" * 60)
    print("LOGIT COLLECTION — this will take significant time on GPU")
    print("=" * 60)
    for draft_short, draft_hf, target_short, target_hf in MODEL_PAIRS:
        for bname, items in ITEMS.items():
            for run in range(N_RUNS):
                print(f"\nCollecting: {target_short} / {bname} / run {run}")
                collect_logits(target_hf, items, bname, run)
                print(f"Collecting: {draft_short} / {bname} / run {run}")
                collect_logits(draft_hf, items, bname, run)
else:
    print("Logit collection skipped (RUN_LOGIT_COLLECTION=False). Using cached .npz files.")

## § 4  Cache Loading & Validation
Validates that all required logit caches exist before proceeding.

In [ ]:
EPS = 1e-12   # numerical epsilon

def load_logit_cache(model_name: str, benchmark: str, run: int = 0) -> dict:
    """
    Load a logit cache from disk.

    Returns a dict with keys:
      item_ids, gen_ids, topk_ids, topk_logprobs
    Each *_ids / *_logprobs array is object-dtype, shape (n_items,),
    where each element is an ndarray of shape (n_steps, TOP_K_STORE).
    """
    path = get_logit_cache_path(model_name, benchmark, run)
    if not path.exists():
        raise FileNotFoundError(
            f"Cache not found: {path}\n"
            "Run § 3 with RUN_LOGIT_COLLECTION=True to generate it."
        )
    data = np.load(path, allow_pickle=True)
    version = int(data["cache_version"])
    if version != LOGIT_CACHE_VERSION:
        raise ValueError(
            f"Cache version mismatch in {path.name}: "
            f"expected {LOGIT_CACHE_VERSION}, got {version}"
        )
    return {
        "item_ids":      data["item_ids"],
        "gen_ids":       data["gen_ids"],
        "topk_ids":      data["topk_ids"],
        "topk_logprobs": data["topk_logprobs"],
    }


# Validate all caches are present
print("Validating logit caches ...")
missing = []
for draft_short, draft_hf, target_short, target_hf in MODEL_PAIRS:
    for bname in ITEMS:
        for run in range(N_RUNS):
            for mname, mhf in [(draft_short, draft_hf), (target_short, target_hf)]:
                p = get_logit_cache_path(mhf, bname, run)
                status = "OK" if p.exists() else "MISSING"
                if status == "MISSING":
                    missing.append(p)
                print(f"  [{status}] {p.name}")

if missing:
    print(f"\n⚠  {len(missing)} cache(s) missing. Set RUN_LOGIT_COLLECTION=True in § 1.")
else:
    print("\n✓  All caches present.")

## § 5  Decoding Simulation
Replays stored logits under each decoding condition **without re-running any model forward passes**.  
This makes all condition comparisons exact and paired at the item level.

In [ ]:
def topk_to_full_dist(topk_ids: np.ndarray, topk_logprobs: np.ndarray,
                      vocab_size: int, temperature: float = 1.0) -> np.ndarray:
    """
    Reconstruct a sparse probability distribution from top-k logprobs.

    Applies temperature scaling before softmax. Probability mass not covered
    by the top-k tokens is distributed uniformly over the remaining vocabulary.

    Returns:
        probs: ndarray of shape (vocab_size,), sums to 1.
    """
    # Re-scale logprobs by temperature then normalise over top-k tokens
    scaled = topk_logprobs / temperature
    scaled -= scaled.max()        # numerical stability
    exp    = np.exp(scaled)
    topk_probs = exp / (exp.sum() + EPS)

    probs = np.zeros(vocab_size, dtype=np.float64)
    probs[topk_ids] = topk_probs
    total = probs.sum()
    if total < 1.0 - 1e-6 and vocab_size > len(topk_ids):
        remainder = 1.0 - total
        mask = np.ones(vocab_size, dtype=bool)
        mask[topk_ids] = False
        probs[mask] += remainder / mask.sum()
    probs = np.clip(probs, 0, None)
    probs /= (probs.sum() + EPS)
    return probs


def simulate_speculative_step(
    p_target: np.ndarray,
    p_draft:  np.ndarray,
    proposed_token: int,
    epsilon: float,
    rng: np.random.Generator,
) -> tuple[int, float]:
    """
    Simulate one speculative decoding acceptance step.

    Args:
        p_target:       Target model probability distribution (vocab_size,).
        p_draft:        Draft model probability distribution (vocab_size,).
        proposed_token: Token proposed by the draft model.
        epsilon:        Acceptance threshold relaxation (0.0 = exact).
        rng:            NumPy random generator.

    Returns:
        (accepted_token, accept_rate) where accept_rate is the per-step
        acceptance probability used for diagnostics.
    """
    pt = p_target[proposed_token]
    pd = p_draft[proposed_token] + EPS
    ratio = pt / pd

    if epsilon == 0.0:
        # Exact speculative decoding: accept with probability min(1, p_t / p_d)
        threshold = min(1.0, ratio)
    else:
        # Approximate: accept if ratio >= (1 - epsilon)
        threshold = 1.0 if ratio >= (1.0 - epsilon) else 0.0

    u = rng.uniform()
    if u < threshold:
        return proposed_token, float(threshold)
    else:
        # Rejection: sample from corrected distribution
        corrected = np.maximum(p_target - p_draft, 0.0)
        s = corrected.sum()
        if s < EPS:
            corrected = p_target.copy()
            s = corrected.sum()
        corrected /= s
        return int(rng.choice(len(corrected), p=corrected)), float(threshold)


def simulate_condition(
    target_cache: dict,
    draft_cache:  dict,
    item_idx:     int,
    epsilon:      float,
    temperature:  float,
    rng:          np.random.Generator,
) -> tuple[list[int], float]:
    """
    Replay stored logits under a given decoding condition.

    Args:
        target_cache: Cache dict from load_logit_cache for the target model.
        draft_cache:  Cache dict from load_logit_cache for the draft model.
        item_idx:     Index of the item within the cache.
        epsilon:      Speculative acceptance threshold (None → temperature-only replay).
        temperature:  Temperature for logit rescaling.
        rng:          NumPy random generator.

    Returns:
        (token_ids, mean_accept_rate)
    """
    target_topk_ids  = target_cache["topk_ids"][item_idx]   # (steps, K)
    target_topk_lp   = target_cache["topk_logprobs"][item_idx]
    draft_topk_ids   = draft_cache["topk_ids"][item_idx]
    draft_topk_lp    = draft_cache["topk_logprobs"][item_idx]
    draft_gen_ids    = draft_cache["gen_ids"][item_idx]      # draft's original tokens

    n_steps  = min(len(target_topk_ids), len(draft_topk_ids), len(draft_gen_ids))
    vocab_size = int(target_topk_ids.max()) + 1
    vocab_size = max(vocab_size, int(draft_topk_ids.max()) + 1)

    token_ids, accept_rates = [], []
    for step in range(n_steps):
        p_t = topk_to_full_dist(
            target_topk_ids[step], target_topk_lp[step], vocab_size, temperature
        )
        p_d = topk_to_full_dist(
            draft_topk_ids[step], draft_topk_lp[step], vocab_size, temperature
        )
        proposed = int(draft_gen_ids[step])

        if epsilon is None:
            # Temperature replay: sample directly from rescaled target distribution
            tok = int(rng.choice(vocab_size, p=p_t))
            ar  = 1.0
        else:
            tok, ar = simulate_speculative_step(p_t, p_d, proposed, epsilon, rng)

        token_ids.append(tok)
        accept_rates.append(ar)

    mean_ar = float(np.mean(accept_rates)) if accept_rates else 1.0
    return token_ids, mean_ar


def use_cached_generation(
    target_cache: dict,
    draft_cache:  dict,
    item_idx:     int,
    condition:    str,
) -> tuple[list[int], float]:
    """
    Return the pre-generated token sequence for baseline (A) or exact speculation (B).

    For B, also computes the expected per-token acceptance rate without
    changing any output tokens (acceptance is deterministic here because
    the generation was not altered).
    """
    gen_ids = list(target_cache["gen_ids"][item_idx].astype(int))

    if condition == "A":
        return gen_ids, 1.0

    # Condition B: compute token-level accept rates for diagnostics
    target_topk_ids = target_cache["topk_ids"][item_idx]
    target_topk_lp  = target_cache["topk_logprobs"][item_idx]
    draft_topk_ids  = draft_cache["topk_ids"][item_idx]
    draft_topk_lp   = draft_cache["topk_logprobs"][item_idx]
    draft_gen_ids   = draft_cache["gen_ids"][item_idx]

    n_steps = min(len(target_topk_ids), len(draft_topk_ids))
    vocab_size = max(int(target_topk_ids.max()), int(draft_topk_ids.max())) + 1

    accept_rates = []
    for step in range(n_steps):
        p_t = topk_to_full_dist(target_topk_ids[step], target_topk_lp[step], vocab_size)
        p_d = topk_to_full_dist(draft_topk_ids[step], draft_topk_lp[step],  vocab_size)
        proposed = int(draft_gen_ids[step])
        ar = min(1.0, (p_t[proposed] / (p_d[proposed] + EPS)))
        accept_rates.append(ar)

    return gen_ids, float(np.mean(accept_rates)) if accept_rates else 0.0


print("Simulation functions defined.")

In [ ]:
# ── Scoring helpers ────────────────────────────────────────────────────────────

def extract_number(text: str) -> float | None:
    """Extract the first number from a string (for GSM8K scoring)."""
    text = text.replace(",", "")
    match = re.search(r"[-+]?\d*\.?\d+", text)
    return float(match.group()) if match else None


def extract_mcq_choice(text: str) -> str | None:
    """Extract the first A/B/C/D letter from a model output string."""
    text = text.strip().upper()
    match = re.match(r"^([A-D])", text)
    if match:
        return match.group(1)
    match = re.search(r"\b([A-D])\b", text)
    return match.group(1) if match else None


def score_gsm8k(predicted_ids: list[int], item: dict, tokenizer) -> int:
    """Return 1 if the predicted number matches the gold answer, else 0."""
    decoded = tokenizer.decode(predicted_ids, skip_special_tokens=True)
    pred = extract_number(decoded)
    gold = extract_number(item["answer"])
    if pred is None or gold is None:
        return 0
    return int(abs(pred - gold) < 1e-3)


def score_mcq(predicted_ids: list[int], item: dict, tokenizer) -> int:
    """Return 1 if the first MCQ letter in the output matches the gold answer."""
    decoded = tokenizer.decode(predicted_ids, skip_special_tokens=True)
    pred = extract_mcq_choice(decoded)
    return int(pred == item["answer"]) if pred else 0


def score_item(predicted_ids: list[int], item: dict, tokenizer) -> int:
    """Dispatch to the appropriate scoring function based on item format."""
    if item["format"] == "freeform":
        return score_gsm8k(predicted_ids, item, tokenizer)
    return score_mcq(predicted_ids, item, tokenizer)


print("Scoring helpers defined.")

In [ ]:
# ── Main simulation loop ───────────────────────────────────────────────────────
# Iterates over model pairs × benchmarks × runs × conditions.
# Builds raw_df (all item × condition rows) and draft_df (draft standalone scores).

rows, draft_rows, gap_rows = [], [], []

for draft_short, draft_hf, target_short, target_hf in MODEL_PAIRS:
    pair_label = f"{draft_short}→{target_short}"
    print(f"\nModel pair: {pair_label}")

    for bname, items in ITEMS.items():
        print(f"  Benchmark: {bname} ({len(items)} items)")

        tokenizer = get_tokenizer(target_hf)

        for run in range(N_RUNS):
            target_cache = load_logit_cache(target_hf, bname, run)
            draft_cache  = load_logit_cache(draft_hf,  bname, run)

            # ── Draft model standalone accuracy ────────────────────────────────
            draft_tokenizer = get_tokenizer(draft_hf)
            draft_scores = []
            for idx, item in enumerate(items):
                d_ids = list(draft_cache["gen_ids"][idx].astype(int))
                s = score_item(d_ids, item, draft_tokenizer)
                draft_rows.append({
                    "model_pair": pair_label, "benchmark": bname, "run": run,
                    "item_id": item["id"], "correct": s, "construct": item["construct"],
                })
                draft_scores.append(s)

            # ── All decoding conditions ─────────────────────────────────────────
            for cond_key, cond_cfg in CONDITIONS.items():
                epsilon    = cond_cfg["epsilon"]
                temperature = cond_cfg["temperature"]
                cond_type  = cond_cfg["type"]
                run_seed   = SEED + run * 1000 + list(CONDITIONS.keys()).index(cond_key)
                rng        = np.random.default_rng(run_seed)

                for idx, item in enumerate(items):
                    if cond_key in ("A", "B"):
                        tok_ids, ar = use_cached_generation(
                            target_cache, draft_cache, idx, cond_key
                        )
                    else:
                        tok_ids, ar = simulate_condition(
                            target_cache, draft_cache, idx, epsilon, temperature, rng
                        )

                    correct = score_item(tok_ids, item, tokenizer)
                    rows.append({
                        "model_pair":   pair_label,
                        "benchmark":    bname,
                        "condition":    cond_key,
                        "cond_type":    cond_type,
                        "run":          run,
                        "item_id":      item["id"],
                        "correct":      correct,
                        "token_ids":    tok_ids,
                        "n_tokens":     len(tok_ids),
                        "accept_rate":  ar,
                        "temperature":  temperature,
                        "construct":    item["construct"],
                        "benchmark_n_items": len(items),
                    })

            # ── Accuracy gap (target minus draft) ──────────────────────────────
            target_baseline = [
                r["correct"] for r in rows
                if r["model_pair"] == pair_label
                and r["benchmark"] == bname
                and r["run"] == run
                and r["condition"] == "A"
            ]
            target_acc = np.mean(target_baseline) if target_baseline else float("nan")
            draft_acc  = np.mean(draft_scores)
            gap_rows.append({
                "model_pair": pair_label, "benchmark": bname, "run": run,
                "target_acc": target_acc,
                "draft_acc":  draft_acc,
                "accuracy_gap": target_acc - draft_acc,
            })

raw_df    = pd.DataFrame(rows)
draft_df  = pd.DataFrame(draft_rows)
gap_df    = pd.DataFrame(gap_rows)

# Save raw outputs
raw_df.drop(columns=["token_ids"]).to_csv(RESULT_DIR / "raw_results.csv",    index=False)
draft_df.to_csv(RESULT_DIR / "draft_standalone.csv", index=False)
gap_df.to_csv(RESULT_DIR / "accuracy_gaps.csv",      index=False)

print(f"\nSimulation complete: {len(raw_df)} rows")
print(f"  Expected: {sum(len(v) for v in ITEMS.values()) * len(MODEL_PAIRS) * N_RUNS * len(CONDITIONS)}")
print(gap_df.to_string(index=False))

## § 6  Build Analysis Dataset
Construct `analysis_df` by computing paired metrics (score flips, token changes, accuracy deltas)
relative to baseline condition A. All three benchmarks use their full observed item sets.

In [ ]:
def build_analysis_df(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    Join each non-baseline condition back to baseline A to compute
    per-item paired metrics.

    Returns a DataFrame with columns:
      model_pair, benchmark, condition, cond_type, item_id, construct,
      baseline_correct, condition_correct, score_flip, token_changed,
      accept_rate, temperature, n_items, baseline_accuracy,
      accuracy_delta_vs_A, score_flip_rate.
    """
    baseline = (
        raw_df[raw_df["condition"] == "A"]
        [["model_pair", "benchmark", "run", "item_id", "correct", "token_ids", "n_tokens"]]
        .rename(columns={"correct": "baseline_correct",
                         "token_ids": "baseline_token_ids",
                         "n_tokens": "baseline_n_tokens"})
    )

    non_baseline = raw_df[raw_df["condition"] != "A"].copy()
    merged = non_baseline.merge(
        baseline, on=["model_pair", "benchmark", "run", "item_id"], how="left"
    )

    merged["score_flip"]    = (merged["correct"] != merged["baseline_correct"]).astype(int)
    merged["token_changed"] = merged.apply(
        lambda r: int(r["token_ids"] != r["baseline_token_ids"]) if
        (r["token_ids"] is not None and r["baseline_token_ids"] is not None) else 0,
        axis=1,
    )

    # Aggregate to condition level
    cond_level = (
        merged.groupby(["model_pair", "benchmark", "condition", "run"])
        .agg(
            n_items       =("item_id",         "count"),
            accuracy      =("correct",          "mean"),
            baseline_acc  =("baseline_correct", "mean"),
            score_flip_rate=("score_flip",      "mean"),
            token_change_rate=("token_changed", "mean"),
            mean_accept_rate=("accept_rate",    "mean"),
        )
        .reset_index()
    )
    cond_level["accuracy_delta_vs_A"] = cond_level["accuracy"] - cond_level["baseline_acc"]

    # Add cond_type and temperature back
    cond_meta = pd.DataFrame(
        [{"condition": k, "cond_type": v["type"], "temperature": v["temperature"]}
         for k, v in CONDITIONS.items() if k != "A"]
    )
    cond_level = cond_level.merge(cond_meta, on="condition", how="left")

    # Add alpha_threshold
    cond_level["alpha_threshold"] = cond_level["condition"].apply(
        lambda c: 1.0 - CONDITIONS[c]["epsilon"]
        if (c in CONDITIONS and CONDITIONS[c]["epsilon"] is not None) else None
    )

    return merged, cond_level


analysis_df, condition_impact = build_analysis_df(raw_df)

# Also build a simple accuracy summary (all conditions including A)
accuracy_summary = (
    raw_df.groupby(["model_pair", "benchmark", "condition", "run"])
    .agg(n_items=("item_id", "count"), mean=("correct", "mean"))
    .reset_index()
)
accuracy_summary["ci_lo"] = accuracy_summary["mean"]
accuracy_summary["ci_hi"] = accuracy_summary["mean"]

analysis_df.drop(columns=["token_ids", "baseline_token_ids"], errors="ignore").to_csv(
    RESULT_DIR / "analysis_dataset.csv", index=False
)
condition_impact.to_csv(RESULT_DIR / "condition_impact.csv", index=False)
accuracy_summary.to_csv(RESULT_DIR / "a1_summary_final.csv",  index=False)

print(f"Analysis dataset: {len(analysis_df)} rows")
print(condition_impact[["model_pair","benchmark","condition","accuracy","accuracy_delta_vs_A","score_flip_rate"]]
      .to_string(index=False))

## § 7  Validity Diagnostics
Computes mechanism-level predictors of score flips (acceptance rate, JS divergence,
target entropy, etc.) and reports point-biserial correlations.

In [ ]:
MECH_MAX_STEPS = 128   # max generation steps used in mechanism diagnostics

def build_mechanism_diagnostics(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute per-item mechanism statistics from stored top-k logits.

    Computed for each model-pair × benchmark × item:
      accept_rate, mean_js_div_topk, mean_target_entropy_topk,
      mean_exact_reject_prob, mean_target_prob_draft_token
    """
    records = []
    for draft_short, draft_hf, target_short, target_hf in MODEL_PAIRS:
        pair_label = f"{draft_short}→{target_short}"
        for bname, items in ITEMS.items():
            for run in range(N_RUNS):
                try:
                    target_cache = load_logit_cache(target_hf, bname, run)
                    draft_cache  = load_logit_cache(draft_hf,  bname, run)
                except FileNotFoundError:
                    continue

                for idx, item in enumerate(items):
                    tki  = target_cache["topk_ids"][idx]
                    tklp = target_cache["topk_logprobs"][idx]
                    dki  = draft_cache["topk_ids"][idx]
                    dklp = draft_cache["topk_logprobs"][idx]
                    dgen = draft_cache["gen_ids"][idx]
                    n_steps = min(len(tki), len(dki), len(dgen), MECH_MAX_STEPS)

                    if n_steps == 0:
                        continue

                    vocab_size = max(int(tki[:n_steps].max()), int(dki[:n_steps].max())) + 1
                    acc_rates, js_divs, entropies, reject_probs, tgt_probs = [], [], [], [], []

                    for step in range(n_steps):
                        p_t = topk_to_full_dist(tki[step], tklp[step], vocab_size)
                        p_d = topk_to_full_dist(dki[step], dklp[step], vocab_size)
                        tok = int(dgen[step])

                        ar = min(1.0, p_t[tok] / (p_d[tok] + EPS))
                        acc_rates.append(ar)

                        m = 0.5 * (p_t + p_d)
                        js = 0.5 * (np.sum(p_t * np.log(p_t / (m + EPS) + EPS))
                                   + np.sum(p_d * np.log(p_d / (m + EPS) + EPS)))
                        js_divs.append(float(np.clip(js, 0, None)))

                        ent = -np.sum(p_t * np.log(p_t + EPS))
                        entropies.append(float(ent))

                        reject_probs.append(float(np.maximum(p_t - p_d, 0).sum()))
                        tgt_probs.append(float(p_t[tok]))

                    records.append({
                        "model_pair": pair_label, "benchmark": bname,
                        "item_id":    item["id"],  "run": run,
                        "accept_rate":                float(np.mean(acc_rates)),
                        "mean_js_div_topk":            float(np.mean(js_divs)),
                        "mean_target_entropy_topk":    float(np.mean(entropies)),
                        "mean_exact_reject_prob":      float(np.mean(reject_probs)),
                        "mean_target_prob_draft_token":float(np.mean(tgt_probs)),
                    })

    return pd.DataFrame(records)


print("Computing mechanism diagnostics ...")
mech_df = build_mechanism_diagnostics(raw_df)
mech_df.to_csv(RESULT_DIR / "mechanism_diagnostics.csv", index=False)
print(f"  {len(mech_df)} rows")

# ── Score flip frame (item-level, all non-baseline conditions) ─────────────────
score_flip_df = analysis_df.copy()
# Merge in mechanism statistics (from baseline condition context)
mech_merge = mech_df.groupby(["model_pair", "benchmark", "item_id"]).first().reset_index()
MECH_COLS = ["accept_rate", "mean_js_div_topk", "mean_target_entropy_topk",
             "mean_exact_reject_prob", "mean_target_prob_draft_token"]
score_flip_df = score_flip_df.merge(
    mech_merge[["model_pair", "benchmark", "item_id"] + MECH_COLS],
    on=["model_pair", "benchmark", "item_id"], how="left"
)
score_flip_df.drop(columns=["token_ids", "baseline_token_ids"], errors="ignore").to_csv(
    RESULT_DIR / "score_flips.csv", index=False
)

# ── Point-biserial correlations: mechanism predictors vs score flips ───────────
assoc_rows = []
for bname in ITEMS:
    sub = score_flip_df[score_flip_df["benchmark"] == bname]
    for cond in SPEC_CONDITIONS + TEMP_CONDITIONS:
        csub = sub[sub["condition"] == cond].dropna(subset=MECH_COLS + ["score_flip"])
        if len(csub) < 20:
            continue
        for pred in MECH_COLS:
            r, p = pointbiserialr(csub[pred], csub["score_flip"])
            assoc_rows.append({
                "benchmark": bname, "condition": cond,
                "predictor": pred, "point_biserial_r": r, "p": p, "n": len(csub),
            })

assoc_df = pd.DataFrame(assoc_rows)
assoc_df.to_csv(RESULT_DIR / "mechanism_flip_associations.csv", index=False)
print("\nTop mechanism predictors of score flips (MMLU, C_080):")
print(
    assoc_df[(assoc_df["benchmark"]=="mmlu") & (assoc_df["condition"]=="C_080")]
    .sort_values("point_biserial_r")
    [["predictor", "point_biserial_r", "p"]]
    .to_string(index=False)
)

In [ ]:
# ── Figure: spec threshold validity ───────────────────────────────────────────
spec_ci = condition_impact[condition_impact["cond_type"] == "spec"].copy()

fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey=False)
bmarks = ["gsm8k", "hellaswag", "mmlu"]
pairs  = [f"{d}→{t}" for d, _, t, __ in MODEL_PAIRS]

for col, bname in enumerate(bmarks):
    for row, pair in enumerate(pairs):
        ax  = axes[row, col]
        sub = spec_ci[(spec_ci["benchmark"] == bname) & (spec_ci["model_pair"] == pair)]
        ax.axhline(0, color="gray", lw=0.8, ls="--")
        ax.plot(sub["condition"], sub["accuracy_delta_vs_A"], marker="o", label="Δacc")
        ax2 = ax.twinx()
        ax2.plot(sub["condition"], sub["score_flip_rate"], marker="s",
                 color="orange", ls=":", label="flip rate")
        ax.set_title(f"{pair}\n{bname}")
        ax.set_xlabel("Condition")
        ax.set_ylabel("Δ Accuracy")
        ax2.set_ylabel("Flip rate")
        ax.tick_params(axis="x", rotation=45)

fig.suptitle("Approximate Speculation: Accuracy Change & Score Flip Rate", fontsize=13)
plt.tight_layout()
fig.savefig(RESULT_DIR / "fig_spec_threshold_validity.pdf", bbox_inches="tight")
plt.close()
print("Saved fig_spec_threshold_validity.pdf")

# ── Figure: spec vs temperature positive control ───────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for col, bname in enumerate(bmarks):
    for row, pair in enumerate(pairs):
        ax  = axes[row, col]
        sub_s = condition_impact[
            (condition_impact["benchmark"] == bname) &
            (condition_impact["model_pair"] == pair) &
            (condition_impact["cond_type"]  == "spec")
        ]
        sub_t = condition_impact[
            (condition_impact["benchmark"] == bname) &
            (condition_impact["model_pair"] == pair) &
            (condition_impact["cond_type"]  == "temp")
        ]
        ax.plot(range(len(sub_s)), sub_s["accuracy_delta_vs_A"], marker="o", label="Spec")
        ax.plot(range(len(sub_t)), sub_t["accuracy_delta_vs_A"], marker="s",
                color="red", label="Temp")
        ax.axhline(0, color="gray", lw=0.8, ls="--")
        ax.set_title(f"{pair}\n{bname}")
        ax.legend(fontsize=7)

fig.suptitle("Speculative Decoding vs Temperature: Accuracy Change Relative to Baseline", fontsize=12)
plt.tight_layout()
fig.savefig(RESULT_DIR / "fig_spec_vs_temperature_positive_control.pdf", bbox_inches="tight")
plt.close()
print("Saved fig_spec_vs_temperature_positive_control.pdf")

# ── Figure: mechanism flip predictors ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for col, bname in enumerate(bmarks):
    ax  = axes[col]
    sub = assoc_df[(assoc_df["benchmark"] == bname) &
                   (assoc_df["condition"].isin(SPEC_CONDITIONS))]
    if sub.empty:
        continue
    pivot = sub.pivot_table(index="predictor", columns="condition",
                            values="point_biserial_r", aggfunc="mean")
    sns.heatmap(pivot, ax=ax, center=0, cmap="RdBu_r", fmt=".2f", annot=True,
                linewidths=0.5, cbar_kws={"shrink": 0.7})
    ax.set_title(bname)
    ax.set_xlabel("")
    ax.set_ylabel("")

fig.suptitle("Point-biserial r: Mechanism Predictors vs Score Flips", fontsize=12)
plt.tight_layout()
fig.savefig(RESULT_DIR / "fig_mechanism_flip_predictors.pdf", bbox_inches="tight")
plt.close()
print("Saved fig_mechanism_flip_predictors.pdf")

## § 8  Temperature Sensitivity (Positive Control)
Analyzes accuracy as a function of temperature and computes Pearson correlations.
Temperature conditions serve as a positive control for decoding-induced measurement variation.

In [ ]:
temp_rows = []
for pair in gap_df["model_pair"].unique():
    for bname in ITEMS:
        for cond in TEMP_CONDITIONS:
            sub = condition_impact[
                (condition_impact["model_pair"] == pair) &
                (condition_impact["benchmark"]  == bname) &
                (condition_impact["condition"]  == cond)
            ]
            if sub.empty:
                continue
            temp_rows.append({
                "model_pair": pair, "benchmark": bname, "condition": cond,
                "temperature": CONDITIONS[cond]["temperature"],
                "accuracy":    float(sub["accuracy"].values[0]),
            })

temp_df = pd.DataFrame(temp_rows)

# Pearson correlations between temperature and accuracy
corr_rows = []
for pair in temp_df["model_pair"].unique():
    for bname in ITEMS:
        sub = temp_df[(temp_df["model_pair"] == pair) & (temp_df["benchmark"] == bname)]
        if len(sub) < 3:
            continue
        r, p = pearsonr(sub["temperature"], sub["accuracy"])
        corr_rows.append({"model_pair": pair, "benchmark": bname, "r": r, "p": p})

temp_corr = pd.DataFrame(corr_rows)
temp_corr.to_csv(RESULT_DIR / "temp_correlations.csv", index=False)
print("Temperature–accuracy Pearson correlations:")
print(temp_corr.to_string(index=False))

# ── Figure: temperature accuracy curves ───────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for col, bname in enumerate(bmarks):
    for row, pair in enumerate(pairs):
        ax  = axes[row, col]
        sub = temp_df[(temp_df["model_pair"] == pair) & (temp_df["benchmark"] == bname)]
        ax.plot(sub["temperature"], sub["accuracy"], marker="o")
        ax.set_title(f"{pair}\n{bname}")
        ax.set_xlabel("Temperature")
        ax.set_ylabel("Accuracy")

fig.suptitle("Accuracy vs Decoding Temperature", fontsize=13)
plt.tight_layout()
fig.savefig(RESULT_DIR / "fig_temp_accuracy_curve.pdf", bbox_inches="tight")
plt.close()
print("\nSaved fig_temp_accuracy_curve.pdf")

# ── Descriptive G-theory for temperature facet ────────────────────────────────
temp_gtheory = []
for pair in gap_df["model_pair"].unique():
    for bname in ITEMS:
        sub = raw_df[
            (raw_df["model_pair"] == pair) &
            (raw_df["benchmark"]  == bname) &
            (raw_df["condition"].isin(TEMP_CONDITIONS))
        ].copy()
        if sub.empty:
            continue
        pivot = sub.pivot_table(index="item_id", columns="condition",
                                values="correct", aggfunc="mean")
        grand = pivot.values.mean()
        var_item = pivot.mean(axis=1).var(ddof=1)
        var_cond = pivot.mean(axis=0).var(ddof=1)
        resid    = pivot.subtract(pivot.mean(axis=1), axis=0)\
                        .subtract(pivot.mean(axis=0), axis=1) + grand
        var_inter = resid.values.var(ddof=1)
        temp_gtheory.append({
            "model_pair": pair, "benchmark": bname,
            "var_item": var_item, "var_condition": var_cond,
            "var_interaction": var_inter,
        })

tg_df = pd.DataFrame(temp_gtheory)
tg_df.to_csv(RESULT_DIR / "temp_gtheory.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(tg_df))
w = 0.25
ax.bar(x - w, tg_df["var_item"],        w, label="Item")
ax.bar(x,     tg_df["var_condition"],   w, label="Condition")
ax.bar(x + w, tg_df["var_interaction"], w, label="Item×Condition")
ax.set_xticks(x)
ax.set_xticklabels([f"{r.benchmark}\n{r.model_pair.split('→')[1]}" for _, r in tg_df.iterrows()],
                   fontsize=8)
ax.legend()
ax.set_ylabel("Variance component")
ax.set_title("Descriptive G-theory: Temperature Facet")
plt.tight_layout()
fig.savefig(RESULT_DIR / "fig_temp_gtheory.pdf", bbox_inches="tight")
plt.close()
print("Saved fig_temp_gtheory.pdf")

## § 9  Analysis 1 — Aggregate Accuracy & McNemar Tests
Paired McNemar tests for each non-baseline condition vs baseline A,
with Bonferroni correction across the `N_CONDITIONS_NON_BASELINE` comparisons.

In [ ]:
def run_mcnemar_analysis(analysis_df: pd.DataFrame) -> pd.DataFrame:
    """
    Run exact McNemar tests for all non-baseline conditions vs baseline A.

    Returns a DataFrame with columns:
      model_pair, benchmark, condition, n_items, delta, b_cnt, c_cnt,
      p, p_bonf, sig
    """
    rows = []
    for pair in analysis_df["model_pair"].unique():
        for bname in ITEMS:
            sub = analysis_df[
                (analysis_df["model_pair"] == pair) &
                (analysis_df["benchmark"]  == bname)
            ]
            for cond in SPEC_CONDITIONS + TEMP_CONDITIONS:
                csub = sub[sub["condition"] == cond]
                if csub.empty:
                    continue
                b = int(((csub["baseline_correct"] == 1) & (csub["correct"] == 0)).sum())
                c = int(((csub["baseline_correct"] == 0) & (csub["correct"] == 1)).sum())
                n = len(csub)
                delta = float(csub["correct"].mean() - csub["baseline_correct"].mean())

                if b + c == 0:
                    p_raw = 1.0
                else:
                    from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test
                    table = np.array([[n - b - c, b], [c, 0]])  # simplified 2x2
                    # Use exact binomial for small counts
                    from scipy.stats import binom_test
                    p_raw = binom_test(b, b + c, 0.5)

                p_bonf = min(p_raw * N_CONDITIONS_NON_BASELINE, 1.0)
                rows.append({
                    "model_pair": pair, "benchmark": bname, "condition": cond,
                    "n_items": n, "delta": delta, "b_cnt": b, "c_cnt": c,
                    "p": p_raw, "p_bonf": p_bonf, "sig": p_bonf < 0.05,
                })
    return pd.DataFrame(rows)


mcnemar_df = run_mcnemar_analysis(analysis_df)
mcnemar_df.to_csv(RESULT_DIR / "a1_mcnemar_final.csv", index=False)

print("Bonferroni-significant results:")
print(
    mcnemar_df[mcnemar_df["sig"]]
    [["model_pair","benchmark","condition","delta","b_cnt","c_cnt","p","p_bonf"]]
    .to_string(index=False)
)

# ── Figure: accuracy comparison ────────────────────────────────────────────────
fig, axes = plt.subplots(len(pairs), len(bmarks), figsize=(14, 7), sharey=False)
for row, pair in enumerate(pairs):
    for col, bname in enumerate(bmarks):
        ax  = axes[row, col]
        sub = accuracy_summary[
            (accuracy_summary["model_pair"] == pair) &
            (accuracy_summary["benchmark"]  == bname)
        ].sort_values("condition", key=lambda x: x.map({k: i for i, k in enumerate(ALL_CONDITIONS)}))
        colors = ["C0" if c in ("A", "B") else "C1" if c.startswith("C") else "C2"
                  for c in sub["condition"]]
        ax.bar(sub["condition"], sub["mean"], color=colors)
        ax.set_title(f"{pair}\n{bname}", fontsize=8)
        ax.set_ylabel("Accuracy")
        ax.tick_params(axis="x", rotation=60, labelsize=7)
fig.suptitle("Accuracy by Condition (blue=A/B, orange=spec, green=temp)", fontsize=11)
plt.tight_layout()
fig.savefig(RESULT_DIR / "fig1_accuracy_comparison.pdf", bbox_inches="tight")
plt.close()
print("\nSaved fig1_accuracy_comparison.pdf")

## § 10  Analysis 2 — Descriptive Generalizability Theory
Decomposes benchmark score variance into item, condition, and item×condition components
following Cronbach et al. (1972). Run variance is excluded because N_RUNS=1.

In [ ]:
def run_gtheory(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    Descriptive G-theory decomposition: variance components for
    item (i), condition (c), and item x condition (ic) facets.

    Excludes run variance because N_RUNS=1.
    G coefficient = sigma2_item / (sigma2_item + sigma2_condition + sigma2_ic).
    """
    rows = []
    for pair in raw_df["model_pair"].unique():
        for bname in ITEMS:
            sub = raw_df[
                (raw_df["model_pair"] == pair) &
                (raw_df["benchmark"]  == bname)
            ].copy()
            if sub.empty:
                continue
            pivot = sub.pivot_table(index="item_id", columns="condition",
                                    values="correct", aggfunc="mean")
            grand    = pivot.values.mean()
            item_means = pivot.mean(axis=1)
            cond_means = pivot.mean(axis=0)

            var_item = float(item_means.var(ddof=1))
            var_cond = float(cond_means.var(ddof=1))
            residuals = (pivot
                         .subtract(item_means, axis=0)
                         .subtract(cond_means, axis=1) + grand)
            var_ic   = float(residuals.values.var(ddof=1))
            var_total = var_item + var_cond + var_ic
            G = var_item / (var_total + 1e-12)

            rows.append({
                "model_pair":       pair,
                "benchmark":        bname,
                "var_item":         var_item,
                "var_condition":    var_cond,
                "var_cond_x_item":  var_ic,
                "var_error":        0,
                "var_total":        var_total,
                "G_coeff":          G,
            })
    return pd.DataFrame(rows)


gt_df = run_gtheory(raw_df)
gt_df.to_csv(RESULT_DIR / "a2_gtheory_final.csv", index=False)
print(gt_df[["model_pair","benchmark","var_item","var_condition","var_cond_x_item","G_coeff"]]
      .to_string(index=False))

# ── Figure: G-theory bar chart ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(gt_df))
w = 0.25
ax.bar(x - w, gt_df["var_item"],        w, label="Item")
ax.bar(x,     gt_df["var_condition"],   w, label="Condition")
ax.bar(x + w, gt_df["var_cond_x_item"], w, label="Item × Condition")
ax.set_xticks(x)
ax.set_xticklabels(
    [f"{r.benchmark}\n{r.model_pair.split('→')[1]}" for _, r in gt_df.iterrows()],
    fontsize=8
)
ax.legend()
ax.set_ylabel("Variance component")
ax.set_title("Descriptive G-theory: Speculative Decoding Facet (N_RUNS=1)")
plt.tight_layout()
fig.savefig(RESULT_DIR / "fig2_gtheory_comparison.pdf", bbox_inches="tight")
plt.close()
print("\nSaved fig2_gtheory_comparison.pdf")

## § 11  Analysis 3 — Construct Stability
Compares accuracy changes across benchmark constructs under each decoding condition.

In [ ]:
construct_df = condition_impact[[
    "model_pair", "benchmark", "condition", "cond_type",
    "accuracy", "accuracy_delta_vs_A"
]].copy()
# Add baseline rows
baseline_rows = []
for pair in gap_df["model_pair"].unique():
    for bname in ITEMS:
        sub = accuracy_summary[
            (accuracy_summary["model_pair"] == pair) &
            (accuracy_summary["benchmark"]  == bname) &
            (accuracy_summary["condition"]  == "A")
        ]
        if sub.empty:
            continue
        baseline_rows.append({
            "model_pair": pair, "benchmark": bname,
            "condition": "A", "cond_type": "spec",
            "accuracy": float(sub["mean"].values[0]),
            "accuracy_delta_vs_A": 0.0,
        })

construct_df = pd.concat([construct_df, pd.DataFrame(baseline_rows)], ignore_index=True)
construct_df.to_csv(RESULT_DIR / "a4_construct_final.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=False)
for col, bname in enumerate(bmarks):
    ax = axes[col]
    for pair in pairs:
        sub = construct_df[
            (construct_df["benchmark"]  == bname) &
            (construct_df["model_pair"] == pair)
        ]
        cond_order = [c for c in ALL_CONDITIONS if c != "A"]
        sub = sub[sub["condition"].isin(cond_order)]
        sub = sub.set_index("condition").reindex(cond_order)
        ax.plot(range(len(sub)), sub["accuracy_delta_vs_A"],
                marker="o", label=pair.split("→")[1])
    ax.axhline(0, color="gray", lw=0.8, ls="--")
    ax.set_xticks(range(len(cond_order)))
    ax.set_xticklabels(cond_order, rotation=60, fontsize=7)
    ax.set_title(bname)
    ax.set_ylabel("Δ Accuracy vs Baseline")
    ax.legend(fontsize=7)

fig.suptitle("Construct-Level Accuracy Changes Across Decoding Conditions", fontsize=12)
plt.tight_layout()
fig.savefig(RESULT_DIR / "fig4_construct_comparison.pdf", bbox_inches="tight")
plt.close()
print("Saved fig4_construct_comparison.pdf")

## § 12  Analysis 4 — Acceptance Rate Regression
Regresses accuracy change on acceptance rate and draft–target gap
to characterize how score degradation scales with distributional deviation.

In [ ]:
# Build regression dataset: each row is one condition × benchmark × pair
reg_data = condition_impact[
    condition_impact["cond_type"] == "spec"
].copy()
reg_data = reg_data.merge(
    gap_df[["model_pair", "benchmark", "accuracy_gap"]],
    on=["model_pair", "benchmark"], how="left"
)
reg_data["reject_rate"] = 1.0 - reg_data["mean_accept_rate"].fillna(0)

reg_data.to_csv(RESULT_DIR / "acceptance_rate_data.csv", index=False)

# Model 1: Δacc ~ reject_rate
clean = reg_data.dropna(subset=["accuracy_delta_vs_A", "reject_rate"])
if len(clean) >= 5:
    X1 = sm.add_constant(clean["reject_rate"])
    m1 = sm.OLS(clean["accuracy_delta_vs_A"], X1).fit()
    pd.DataFrame({"param": m1.params, "pvalue": m1.pvalues}).to_csv(
        RESULT_DIR / "reg_model1.csv"
    )
    print("Model 1 (Δacc ~ reject_rate):")
    print(m1.summary2().tables[1])

# Model 2: Δacc ~ reject_rate + accuracy_gap (moderation)
clean2 = reg_data.dropna(subset=["accuracy_delta_vs_A", "reject_rate", "accuracy_gap"])
if len(clean2) >= 6:
    X2 = sm.add_constant(clean2[["reject_rate", "accuracy_gap"]])
    m2 = sm.OLS(clean2["accuracy_delta_vs_A"], X2).fit()
    pd.DataFrame({"param": m2.params, "pvalue": m2.pvalues}).to_csv(
        RESULT_DIR / "reg_model2.csv"
    )
    print("\nModel 2 (Δacc ~ reject_rate + accuracy_gap):")
    print(m2.summary2().tables[1])

# ── Figure: acceptance rate moderation ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
for pair in pairs:
    sub = clean[clean["model_pair"] == pair]
    ax.scatter(sub["reject_rate"], sub["accuracy_delta_vs_A"],
               label=pair.split("→")[1], alpha=0.7)
ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.set_xlabel("Rejection rate (1 − accept_rate)")
ax.set_ylabel("Δ Accuracy vs Baseline")
ax.set_title("Accuracy Change vs Rejection Rate")
ax.legend()
plt.tight_layout()
fig.savefig(RESULT_DIR / "fig6_acceptance_moderation.pdf", bbox_inches="tight")
plt.close()
print("\nSaved fig6_acceptance_moderation.pdf")

## § 13  Summary
Prints a concise hypothesis evaluation and lists all output files.

In [ ]:
print("=" * 65)
print("HYPOTHESIS EVALUATION SUMMARY")
print("=" * 65)

exact_rows = mcnemar_df[mcnemar_df["condition"] == "B"]
h1_pass = all(exact_rows["p"] == 1.0)
print(f"\nH1 (Exact spec preserves scores): {'PASS' if h1_pass else 'FAIL'}")
print(f"   McNemar p=1.0 for all pairs: {h1_pass}")

sig_spec = mcnemar_df[
    mcnemar_df["sig"] & mcnemar_df["condition"].isin(SPEC_CONDITIONS)
]
print(f"\nH2 (Approximate spec can shift scores):")
if sig_spec.empty:
    print("   No Bonferroni-significant spec results.")
else:
    print(sig_spec[["model_pair","benchmark","condition","delta","p_bonf"]].to_string(index=False))

print(f"\nH3 (Temperature is a valid positive control):")
sig_temp = mcnemar_df[
    mcnemar_df["sig"] & mcnemar_df["condition"].isin(TEMP_CONDITIONS)
]
print(f"   Bonferroni-significant temperature conditions: {len(sig_temp)}")
if not sig_temp.empty:
    print(sig_temp[["model_pair","benchmark","condition","delta","p_bonf"]].to_string(index=False))

print(f"\nG-Theory (condition variance should be negligible):")
for _, r in gt_df.iterrows():
    frac = r.var_condition / (r.var_total + 1e-12)
    print(f"   {r.model_pair} / {r.benchmark}: var_cond={r.var_condition:.5f} "
          f"({frac*100:.2f}% of total), G={r.G_coeff:.6f}")

print("\n" + "=" * 65)
print("OUTPUT FILES")
print("=" * 65)
for f in sorted(RESULT_DIR.glob("*")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<50} {size_kb:6.1f} KB")